# Prophet Time-Series — Seasonal Adulteration Prediction
Predicts which foods are likely to be adulterated in the coming weeks based on FSSAI violation history + seasonal patterns.
Uses Facebook Prophet (free, open source).

In [ ]:
# pip install prophet pandas matplotlib

from prophet import Prophet
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

print('Prophet version:', Prophet.__version__ if hasattr(Prophet, '__version__') else 'installed')

In [ ]:
# Simulate FSSAI violation data (replace with real DB query in production)
# In production: query scan_records + fssai_violations tables

import numpy as np

# Turmeric violation counts per month (historical pattern)
# Spike in Oct-Nov (Diwali) and March-April (Holi)
np.random.seed(42)
dates = pd.date_range('2020-01-01', '2024-12-01', freq='MS')

def seasonal_violations(dates, base=10, diwali_boost=35, holi_boost=20):
    """Simulate realistic violation counts with seasonal spikes."""
    counts = []
    for d in dates:
        count = base + np.random.poisson(5)
        if d.month in [10, 11]:  count += diwali_boost + np.random.randint(0, 15)
        if d.month in [3]:       count += holi_boost   + np.random.randint(0, 10)
        if d.month in [6,7,8]:   count += 12           + np.random.randint(0,  8)  # monsoon
        counts.append(count)
    return counts

df_turmeric = pd.DataFrame({'ds': dates, 'y': seasonal_violations(dates)})
print(df_turmeric.tail())

In [ ]:
# Train Prophet model
model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False,
    seasonality_mode='multiplicative',  # violations multiply, not add
    changepoint_prior_scale=0.1,
)

# Add Indian festival regressors as special events
festivals = pd.DataFrame({
    'holiday': ['diwali','diwali','holi','holi','navratri','navratri'],
    'ds':      pd.to_datetime(['2022-10-24','2023-11-12','2023-03-08','2024-03-25',
                               '2023-10-15','2024-10-03']),
    'lower_window': [-7,-7,-7,-7,-5,-5],
    'upper_window': [7,7,3,3,5,5],
})
model.add_country_holidays(country_name='IN')

model.fit(df_turmeric)
print('Model trained!')

In [ ]:
# Forecast next 12 months
future   = model.make_future_dataframe(periods=12, freq='MS')
forecast = model.predict(future)

# Plot
fig = model.plot(forecast, figsize=(14,5))
plt.title('Predicted Turmeric Adulteration Reports — Next 12 Months')
plt.xlabel('Date')
plt.ylabel('Expected Violation Count')
plt.tight_layout()
plt.savefig('../data/turmeric_forecast.png', dpi=150)
plt.show()

# Show upcoming spikes
future_only = forecast[forecast['ds'] > pd.Timestamp.now()][['ds','yhat','yhat_lower','yhat_upper']]
future_only['risk'] = future_only['yhat'].apply(lambda x: 'HIGH' if x>45 else 'MEDIUM' if x>25 else 'LOW')
print(future_only.to_string(index=False))

In [ ]:
# Save model and generate prediction API
import joblib, json

joblib.dump(model, '../models/prophet_turmeric.pkl')

# Generate risk calendar (used by the app)
calendar = {}
for _, row in future_only.iterrows():
    month_key = row['ds'].strftime('%Y-%m')
    calendar[month_key] = {
        'predicted_violations': round(row['yhat'], 1),
        'risk': row['risk'],
        'upper': round(row['yhat_upper'], 1),
    }

with open('../models/turmeric_risk_calendar.json', 'w') as f:
    json.dump(calendar, f, indent=2)

print('Saved model + risk calendar')